# **Integrating pre-processed batches** - Healthy Mouse Multiome snRNA and snATAC sequencing)

# Params

In [ ]:
params <- list(
    samples_names = c("Pilot1", "Batch1"),
    samples_directories = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/seurat/individual_preprocessing/results/sobjs",
    sample_id = "Integration_P1-B1",
    res_dir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/seurat/integration/results",
    table_dir = "tables",
    so_dir = "sobjs",
    fig_dir = "figures"
)

In [ ]:
params2 <- list(
    rna_merged_dim = 1:50,
    peaks_merged_dim = 2:50,
    peaks_integration_anchors_dim = 1:50, 
    peaks_integration_dim = 2:50 
)

# Load Packages

In [ ]:
packages <- c("Seurat","Signac", "here",
  "cowplot","tidyverse", "patchwork", "ggplot2", 
  "DoubletFinder",
  "clustree", "sceasy",
  "Matrix", "matrixStats",
  "Herper", "reticulate", "ggVennDiagram")

purrr::walk(packages, library, character.only = TRUE) #character.only to treat every element of 'packages' as character strings representing package names. 

# Check the version of Seurat being used here 
as.character(packageVersion("Seurat")) 

# Set directories

In [ ]:
# set path to results folder, seurat objects folder and tables folder 
res_dir <- here(paste0(params$res_dir))
so_dir <- here(res_dir, params$so_dir)
table_dir <- here(res_dir, params$table_dir)
fig_dir <- here(res_dir, params$fig_dir)

# Create directories if they don't exist
dir.create(file.path(res_dir), showWarnings = TRUE, recursive = TRUE)
dir.create(file.path(so_dir), showWarnings = TRUE, recursive = TRUE)
dir.create(file.path(table_dir), showWarnings = TRUE, recursive = TRUE)
dir.create(file.path(fig_dir), showWarnings = TRUE, recursive = TRUE)

# **Seurat Objects**

In [ ]:
# initialize an empty list to store seurat objects
seurat_objects <- list()

# Loop through each directory and read the corresponding RDS file
for (i in seq_along(params$samples_names)) {
  seurat_obj <- readRDS(
    file.path(
      params$samples_directories,
      paste0(params$samples_names[i], "_seurat_singlets_WNNmultiome.rds")
    )
  )
  DefaultAssay(seurat_obj) <- "RNA"
  seurat_objects[[i]] <- seurat_obj
}



In [ ]:
# Print summary of the loaded Seurat objects
for (i in seq_along(seurat_objects)) {
  cat("Sample", i, ":", params$samples_names[[i]], "\n")
  print(seurat_objects[[i]])
}

## Add hashtags to metadata

In [ ]:
unique(seurat_objects[[1]]$HTO_LibNorm_classification)

In [ ]:
seurat_objects[[1]]$HTO_origin <- recode(
  seurat_objects[[1]]$HTO_LibNorm_classification,
  "tag-3" = "P1_tag3",
  "tag-5" = "P1_tag5",
  "tag-6" = "P1_tag6",
  "tag-7" = "P1_tag7",
  "tag-8" = "P1_tag8",
  "tag-9" = "P1_tag9")

In [ ]:
seurat_objects[[2]]$HTO_origin <- recode(
  seurat_objects[[2]]$HTO_LibNorm_classification,
  "tag-1" = "B1_tag1",
  "tag-3" = "B1_tag3",
  "tag-6" = "B1_tag6",
  "tag-8" = "B1_tag8"
  )

# **RNA**

## **Merge RNA assays first to confirm that Integrating them with Harmony is needed**

Since this relied on DietSeurat, and it doesn't seem to work seamlessly on Version 5, I will re-merge the whole objects and integrate them later. 

In [ ]:
rna_objects <- lapply(seurat_objects, function(obj) {
  DietSeurat(obj, assays = "RNA", counts = TRUE, data = TRUE, scale.data = TRUE)})

In [ ]:
rna_objects2 <- lapply(seurat_objects, function(obj) {
  DietSeurat(
    obj,
    assays = "RNA",
    layers = c("counts", "data", "scale.data")
  )
})

In [ ]:
merged_rna <- merge(
  x = rna_objects[[1]],
  y = rna_objects[-1],
  add.cell.ids = params$samples_names,
  project = "RNACombined",
  merge.data = TRUE
)

### Check QC parameters

In [ ]:
merged_QC <- VlnPlot(
  object = merged_rna,
  features = c("nCount_RNA","nFeature_RNA", "Percent_mito", "Percent_ribo",  "S.Score", "G2M.Score"),
  ncol = 6,
  group.by = "orig.ident2",
  pt.size = 0) + theme(axis.title.x = NULL)

merged_QC


In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNAmerged_QC.png")), merged_QC, width = 20, height = 10, units = "in")

### Quick processing to see where cells lie on the UMAP

In [ ]:
merged_rna <- SCTransform(merged_rna, new.assay.name = "SCT", return.only.var.genes = TRUE)
merged_rna <- RunPCA(merged_rna, assay = "SCT", npcs = 50) 

In [ ]:
e1 <- ElbowPlot(merged_rna, ndims = 50) + labs(title = "B1 + P1 Merged RNA assays \n folowed by SCTransform + RunPCA") 
e1

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNAmerged_ElbowPlot.png")), e1, width = 20, height = 10, units = "in")

In [ ]:
Graphs(merged_rna)

In [ ]:
merged_rna <- FindNeighbors(merged_rna, reduction = "pca", k.param = 30, dims = params2$rna_merged_dim)
merged_rna <-  FindClusters(merged_rna, resolution =  1, graph.name = "SCT_snn") 

In [ ]:
merged_rna <-  RunUMAP(merged_rna, 
                       dims = params2$rna_merged_dim, slot = "data", assay = "SCT", n.neighbors = 30L,
                       reduction.name = 'umapSCTmerged', reduction.key = 'umapSCTmerged_')



In [ ]:
merged_rna_umap <- DimPlot(merged_rna, group.by = "orig.ident2", shuffle = TRUE) + labs(title = "P1 + B1 Merged")
merged_rna_umap

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_sequencing_id, "_RNAmerged_UMAP.png")), merged_rna_umap, width = 20, height = 10, units = "in")


In [ ]:
saveRDS(merged_rna, 
        here(so_dir, paste0(params$sample_id, "_RNAmerged.rds")))

## **Merge and Integrate RNA assays with Harmony Following V5 Tutorial**  

**On my previous analysis, using Seurat V4 I ran Harmony on pre-processed, individual SCT Assays. But this did NOT work with Seurat V5**  
- SCT normalization has been performed independently on each sample  
- Run Harmony on the individual SCT Assays  
    - merge these normalized Seurat objects  
    - find integration features across samples   
    - set variable features manually to integration features     
    - calculate PCs on this merged objects (this workflow reflects recommendations for application of SCTransform)
- Read [this](https://github.com/satijalab/seurat/issues/9557)


**Now processing the integration following the new V5 integration tutorial for SCTransform normalized datasets**  
- Find the tutorial [here](https://satijalab.org/seurat/articles/integration_introduction.html#perform-integration-with-sctransform-normalized-datasets)  

In [ ]:
options(Seurat.object.assay.version = 'v5')

In [ ]:
install.packages('SeuratData')
library(SeuratData)

In [ ]:
# access SCT within each object of the seurat list
for (i in 1:length(seurat_objects)) {
  DefaultAssay(seurat_objects[[i]]) <- "SCT"
}

In [ ]:
# Find most variable features across samples to integrate
int_feat <- SelectIntegrationFeatures(object.list = sct_objects,
                                      nfeatures = 3000)
head(int_feat)

In [ ]:
# merge with merge.dr = TRUE
merged <- merge(
  x = seurat_objects[[1]],
  y = seurat_objects[-1],
  add.cell.ids = params$samples_names,
  project = "Harmony_Integration",
  merge.data = TRUE,
  merge.dr = TRUE)

merged

In [ ]:
DefaultAssay(merged2) <- "RNA"

In [ ]:
merged2 <- SCTransform(merged2)

In [ ]:
merged2 <- RunPCA(merged2, assay = "SCT", npcs = 50)

In [ ]:
ep <- ElbowPlot(merged2, ndims = 50) + labs(title = "B1 + P1 Merged RNA assays V5 pipeline \n folowed by SCTransform + RunPCA")
ep

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNAmerged_V5pipeline_ElbowPlot.png")), ep, width = 20, height = 10, units = "in")

In [ ]:
merged2 <- FindNeighbors(merged2, reduction = "pca", k.param = 30, dims = params2$rna_merged_dim)

In [ ]:
# graph.name - Name of graph to use for the clustering algorithm
merged2 <- FindClusters(merged2, resolution =  1, graph.name = "SCT_snn", cluster.name = "unintegrated_SCT_clusters") 

In [ ]:
merged2 <- RunUMAP(merged2, 
                       dims = params2$rna_merged_dim, slot = "data", assay = "SCT", n.neighbors = 30L,
                       reduction.name = 'umap.unintegrated.SCT', reduction.key = 'UMAP.unintegrated.SCT_')

# reduction.name - Name to store dimensional reduction under in the Seurat object
# reduction.key - dimensional reduction key, specifies the string before the number for the dimension names. UMAP by default


In [ ]:
merged_umap <- DimPlot(merged2, reduction = "umap.unintegrated.SCT",
                       group.by = "orig.ident2", shuffle = TRUE) + labs(title = "P1 + B1 Merged V5 pipeline \n Unintegrated")
merged_umap

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNAmerged_UMAP_V5pipeline.png")), merged_umap, width = 20, height = 15, units = "in")

**Perform Integration**

In [ ]:
merged2 <- IntegrateLayers(object = merged2,
                          method = HarmonyIntegration,
                          orig.reduction = "pca",
                          normalization.method = "SCT",
                          new.reduction = "harmony.SCT")

**When normalizing the dataset with SCT, Seurat stores the results differently compared to LogNorm.**    
**The output is already a single, integrated SCT assay. Therefore, thre is no need to join layers.**

In [ ]:
merged2 <- FindNeighbors(merged2, reduction = "harmony.SCT", k.param = 30, 
                         dims = params2$rna_merged_dim, 
                         graph.name = c("harmonySCT_nn", "harmonySCT_snn"))

In [ ]:
Graphs(merged2)

In [ ]:
merged2 <- FindClusters(merged2, 
                        resolution = c(0.2, 0.5, 0.8, 1), 
                        graph.name = "harmonySCT_snn") 

In [ ]:
merged2 <- RunUMAP(merged2, 
                       dims = params2$rna_merged_dim, slot = "data", assay = "SCT", n.neighbors = 30L,
                       reduction = "harmony.SCT",
                       reduction.name = 'umap.harmony.SCT', reduction.key = 'UMAP.harmony.SCT_')

# reduction.name - Name to store dimensional reduction under in the Seurat object
# reduction.key - dimensional reduction key, specifies the string before the number for the dimension names. UMAP by default

In [ ]:
Reductions(merged2)

In [ ]:
integrated_umap <- DimPlot(merged2, group.by = "orig.ident2", 
                           shuffle = TRUE, reduction = "umap.harmony.SCT") + labs(title = "P1 + B1 Integrated  \n Harmony V5 pipeline")
integrated_umap

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_RNAIntegratedHarmony_UMAP_V5pipeline.png")), merged_umap + integrated_umap, width = 20, height = 10, units = "in")


In [ ]:
saveRDS(merged2, 
        here(so_dir, paste0(params$sample_id, "_SCT_Integrated_Harmony_V5.rds")))

# **ATAC**


In [ ]:
merged2 <- readRDS(
        here(so_dir, paste0(params$sample_id, "_SCT_Integrated_Harmony_V5.rds")))

In [ ]:
DefaultAssay(merged2) <- "peaks"

**After running the RNA integration, even the Peaks assay's layers got merged.   
Therefore I will have to integrate Peaks separately and then join the objects to run WNN.**

### Access peaks assay

In [ ]:
for (i in 1:length(seurat_objects)) {
  DefaultAssay(seurat_objects[[i]]) <- "peaks"
}

### Create a unified set of peaks

In [ ]:
# Extract the ranges from the peaks assay for each object
peaks_ranges <- lapply(seurat_objects, function(obj) {
  obj@assays$peaks@ranges
})

In [ ]:
peaks_ranges

In [ ]:
# create reduced set of peaks
peaks_combined <- GenomicRanges::reduce(do.call(c, peaks_ranges))
peaks_combined

### Filter "bad" peaks based on the following threshold

In [ ]:
# BiocManager::install("BiocGenerics")
library(BiocGenerics)

In [ ]:
peak_widths <- width(peaks_combined)
head(peak_widths)

In [ ]:
params3 <- list(
    min_peak_width = 20,
    max_peak_witdh = 10000)

In [ ]:
width_threshold_comparison <- data.frame(
  Condition = c("All Reduced Peaks", 
                "Peaks < 20", 
                "Peaks > 10000", 
                "Peaks within threshold"),
  Count = c(length(peak_widths), 
            sum(peak_widths < params3$min_peak_width), 
            sum(peak_widths > params3$max_peak_witdh), 
            sum(peak_widths > params3$min_peak_width & 
                peak_widths < params3$max_peak_witdh))
)
width_threshold_comparison

### Quantify peaks for each seurat obj in the list

In [ ]:
# create function to quantify peaks for each seurat obj in my list
quantify_peaks <- function(seurat_obj, peaks_combined) {
  fragments <- seurat_obj@assays$peaks@fragments
  cat("Quantifying peaks using 'peaks_combined' for sample:", unique(seurat_obj$orig.ident2), "\n")
  counts <- FeatureMatrix(fragments = fragments,
                          features = peaks_combined,
                          cells = colnames(seurat_obj))
  return(counts)
}

In [ ]:
peaks_reduced_counts <- list() # list to store results

for (i in seq_along(seurat_objects)){
  peaks_reduced_counts[[i]] <- quantify_peaks(seurat_objects[[i]], peaks_combined) 
}


In [ ]:
head(peaks_reduced_counts)

In [ ]:
# create function to create a reducedPeaks assay to each obj in my list
create_reduced_assay <- function(seurat_obj, peaks_reduced_counts) {
  fragments <- seurat_obj@assays$peaks@fragments
  annotation <- seurat_obj@assays$peaks@annotation
  cat("Creating 'reducedPeaks' assay for sample:", unique(seurat_obj$orig.ident), "\n")
  seurat_obj[["reducedPeaks"]] <- CreateChromatinAssay(counts = peaks_reduced_counts, fragments = fragments, annotation = annotation)
  return(seurat_obj)
}

for (i in seq_along(seurat_objects)){
  seurat_objects[[i]] <- create_reduced_assay(seurat_objects[[i]], peaks_reduced_counts[[i]])
}


In [ ]:
# extract the dimensions of each seurat object with raw macs2 peaks and reduced peaks
dimensions_peaks_list <- data.frame()

for (i in seq_along(seurat_objects)) {
  sample_name <- as.character(unique(seurat_objects[[i]]$orig.ident2))
  peaks_dim <- dim(seurat_objects[[i]]@assays$peaks)
  reduced_peaks_dim <- dim(seurat_objects[[i]]@assays$reducedPeaks)
  
  cat("Processing sample:", sample_name, "\n")
  cat("Peaks dimensions:", peaks_dim, "\n")
  cat("ReducedPeaks dimensions:", reduced_peaks_dim, "\n")
  
  dimensions_peaks_list <- rbind(dimensions_peaks_list, data.frame(
    Sample = sample_name,
    Peaks_Features = peaks_dim[1],
    Peaks_Cells = peaks_dim[2],
    ReducedPeaks_Features = reduced_peaks_dim[1],
    ReducedPeaks_Cells = reduced_peaks_dim[2],
    stringsAsFactors = FALSE
  ))
}
print(dimensions_peaks_list)

In [ ]:
write.csv(dimensions_peaks_list, row.names = FALSE,
          file.path(table_dir, paste0(params$sample_id, "_Peaks-vs-reducedPeaks-dimensions.csv")))

### Merge based on reduced peaks

In [ ]:
for (i in 1:length(seurat_objects)) {
  DefaultAssay(seurat_objects[[i]]) <- "reducedPeaks"}

In [ ]:
{
    merged_peaks_reduced <- merge(
  x = seurat_objects[[1]],
  y = seurat_objects[-1],
  project = "ReducedPeaksMerged",
  add.cell.ids = params$samples_names,
  merge.data = TRUE)
    
    saveRDS(merged_peaks_reduced, 
        here(so_dir, paste0(params$sample_id, "_reducedPeaks_merged.rds")))
    }


In [ ]:
merged_peaks_reduced <- readRDS(
        here(so_dir, paste0(params$sample_id, "_reducedPeaks_merged.rds")))

In [ ]:
merged_peaks_reduced_QC <- VlnPlot(
  object = merged_peaks_reduced,
  features = c("nCount_peaks", "nFeature_peaks", "peak_region_fragments", "TSS.enrichment", "nucleosome_signal", "TSS.enrichment", "nucleosome_signal", "pct_reads_in_peaks", "blacklist_fraction"),
  ncol = 7,
  group.by = "orig.ident2",
  pt.size = 0) + theme(axis.title.x = NULL)
merged_peaks_reduced_QC

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_reducedPeaks_merged_QC.png")), merged_peaks_reduced_QC, width = 25, height = 15, units = "in")

### Processing Merged before integration

In [ ]:
# quick processing to check UMAP before integration
merged_peaks_reduced <-  merged_peaks_reduced %>%
  FindTopFeatures(min.cutoff = "q0", assay = "reducedPeaks") %>%
  RunTFIDF(assay = "reducedPeaks") %>%
  RunSVD(assay = "reducedPeaks", n = 50) 

In [ ]:
e3 <- ElbowPlot(merged_peaks_reduced, ndims = 50, reduction = "lsi") + labs(title = "Elbow Plot - Merged reducedPeaks Assay \n after FindTopFeatures + TFIDF + SVD")
e3

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_reducedPeaks_merged_ElbowPlot.png")), e3, width = 20, height = 10, units = "in")

In [ ]:
merged_peaks_reduced <- merged_peaks_reduced %>% 
  FindNeighbors(reduction = "lsi", dims = params2$peaks_merged_dim) %>%
  FindClusters(resolution = 1)

In [ ]:
merged_peaks_reduced <- FindClusters(merged_peaks_reduced, resolution = 1)


In [ ]:
merged_peaks_reduced <- RunUMAP(merged_peaks_reduced, 
                             reduction = 'lsi', 
                             dims = params2$peaks_merged_dim, 
                             reduction.name = "umap.reducedpeaks.merged", 
                             reduction.key = "UMAP.reducedpeaks.merged")


In [ ]:
reducedPeaks_umap <- DimPlot(merged_peaks_reduced, group.by = "orig.ident2", shuffle = TRUE) + labs(title = "reducedPeaks Merged - Healthy P1 + B1")
reducedPeaks_umap

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_reducedPeaks_merged_UMAP.png")), 
       reducedPeaks_umap,
       width = 30, height = 15, units = "in")

In [ ]:
saveRDS(merged_peaks_reduced, here(so_dir, paste0(params$sample_id, "_reducedPeaks_merged_processed.rds")))

## Integrated merged reduced peaks diet

In [ ]:
merged_peaks_reduced <- 
readRDS(here(so_dir, paste0(params$sample_id, "_reducedPeaks_merged_processed.rds")))

In [ ]:
merged_peaks_reduced

In [ ]:
merged_list <- SplitObject(merged_peaks_reduced, split.by = "orig.ident2")

In [ ]:
params2$peaks_integration_anchors_dim

In [ ]:
peaks.integration.anchors <- FindIntegrationAnchors(
  object.list = merged_list,
  anchor.features = rownames(merged_peaks_reduced),
  reduction = "rlsi",
  dims = params2$peaks_integration_anchors_dim,
  k.anchor = 5, k.filter = 200, k.score = 30,
  max.features = 200, scale = TRUE, n.trees = 50
)

In [ ]:
anchor.features <- peaks.integration.anchors@anchor.features
head(anchor.features)
length(anchor.features)

In [ ]:
# integrate
peaks_reduced_integrated <- IntegrateEmbeddings(
  anchorset = peaks.integration.anchors,
  reductions = merged_peaks_reduced[['lsi']],
  new.reduction.name = "integrated_lsi",
  dims.to.integrate = params2$peaks_integration_dim
)

In [ ]:
head(Embeddings(peaks_reduced_integrated[["integrated_lsi"]]))

In [ ]:
dim(Embeddings(peaks_reduced_integrated[["integrated_lsi"]]))

In [ ]:
# process integrated obj
peaks_reduced_integrated <- RunUMAP(peaks_reduced_integrated, 
                                    reduction = "integrated_lsi", 
                                    dims = 1:49, 
                                    reduction.name = "umap.reducedpeaks.integrated",
                                    reduction.key = "UMAP.reducedpeaks.integrated")

In [ ]:
# process integratedo object
{
peaks_reduced_integrated <-  peaks_reduced_integrated %>%
        FindNeighbors(reduction = "integrated_lsi", dims = 1:49, graph.name = c("peaks_reduced_integrated_nn", "peaks_reduced_integrated_snn")) %>%
        FindClusters(resolution = c(0.2, 0.5, 0.8, 1), graph.name = "peaks_reduced_integrated_snn")

    saveRDS(peaks_reduced_integrated, here(so_dir, paste0(params$sample_sequencing_id, "_seurat_reducedPeaks_integrated.rds")))
    }

In [ ]:
peaks_reduced_integrated

In [ ]:
peaks_reduced_integrated_umap <- DimPlot(peaks_reduced_integrated,
                                         group.by = "orig.ident2", shuffle = TRUE) + labs(title = "Peaks Reduced Integrated - Healthy P1 + B1")
peaks_reduced_integrated_umap

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_sequencing_id, "_reducedPeaks_integrated-UMAP.png")), 
       peaks_reduced_integrated_umap, 
       width = 30, height = 15, units = "in")


In [ ]:
# visualize clusters
c21 <- DimPlot(peaks_reduced_integrated, group.by = "peaks_reduced_integrated_snn_res.0.2", label = TRUE, pt.size = 1) + labs(title = "Clusters reduced peaks integrated - res.0.2 - Healthy P1+B1", paste0(params$peaks_integration_dim, "PCs"))
c22 <- DimPlot(peaks_reduced_integrated, group.by = "peaks_reduced_integrated_snn_res.0.5", label = TRUE, pt.size = 1) + labs(title = "Clusters reduced peaks integrated - res.0.5 - Healthy P1+B1", paste0(params$peaks_integration_dim, "PCs"))
c23 <- DimPlot(peaks_reduced_integrated, group.by = "peaks_reduced_integrated_snn_res.0.8", label = TRUE, pt.size = 1) + labs(title = "Clusters reduced peaks integrated - res.0.8 - Healthy P1+B1", paste0(params$peaks_integration_dim, "PCs"))
c24 <- DimPlot(peaks_reduced_integrated, group.by = "peaks_reduced_integrated_snn_res.1", label = TRUE, pt.size = 1) + labs(title = "Clusters reduced peaks integrated - res.1 - Healthy P1+B1", paste0(params$peaks_integration_dim, "PCs"))
c21 + c22 + c23 + c24

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_sequencing_id, "_reducedPeaks-integrated-clusters-resolutions-comparison.png")), c21+c22+c23+c24, width = 25, height = 17, units = "in")

# **WNN**
See explanation of this integration [here](https://satijalab.org/seurat/articles/weighted_nearest_neighbor_analysis)

In [ ]:
# load integrated RNA
harmony_sct <- readRDS( 
        here(so_dir, paste0(params$sample_id, "_SCT_Integrated_Harmony_V5.rds")))

In [ ]:
peaks_reduced_integrated

In [ ]:
# Add ATAC integrated assay to the RNA integrated one
w <- harmony_sct

w[["reducedPeaks"]] <- peaks_reduced_integrated[["reducedPeaks"]]
w[["integrated_lsi"]] <- peaks_reduced_integrated[["integrated_lsi"]]
w[["umap.reducedpeaks.integrated"]] <- peaks_reduced_integrated[["umap.reducedpeaks.integrated"]]
w


In [ ]:
params2

In [ ]:
# Run WNN 
w <- FindMultiModalNeighbors(
  object = w,
  reduction.list = list("harmony.SCT", "integrated_lsi"),
  dims.list = list(params$rna_merged_dim, params$peaks_merged_dim),
  verbose = TRUE)

In [ ]:
# Run WNN 
w <- FindMultiModalNeighbors(
  object = w,
  reduction.list = list("harmony.SCT", "integrated_lsi"),
  dims.list = list(1:50, 1:49),
  verbose = TRUE)

In [ ]:
Reductions(w)
Graphs(w)

In [ ]:
{
w <- RunUMAP(
  object = w, 
  nn.name = "weighted.nn", 
  verbose = TRUE,
  reduction.name = "jointumap",
  reduction.key = "jointUMAP_")

w <- FindClusters(
  w, 
  graph.name = "wsnn", 
  algorithm = 3, verbose = FALSE, resolution = c(0.2, 0.5, 0.8, 1))

# save objetct
saveRDS(w, here(so_dir, paste0(params$sample_sequencing_id, "_seurat-wnn-integrated2.RDS")))

}

In [ ]:
w <- readRDS(here(so_dir, paste0(params$sample_sequencing_id, "_seurat-wnn-integrated2.RDS")))

In [ ]:
w

In [ ]:
# plot RNA, ATAC and WNN
w1 <- DimPlot(w, reduction = "umap.harmony.SCT", label = TRUE, label.size = 3, repel = TRUE) + ggtitle("Integrated RNA - Healthy P1+B1")
w2 <- DimPlot(w, reduction = "umap.reducedpeaks.integrated",  label = TRUE, label.size = 3, repel = TRUE) + ggtitle("Integrated ATAC - Healthy P1+B1")
w1+w2

In [ ]:
as.character(0:max(as.numeric(w@meta.data$wsnn_res.1)))
w3 <- DimPlot(w, reduction = "jointumap", label = TRUE, label.size = 3, repel = TRUE, group.by = "wsnn_res.1") + ggtitle("Integrated WNN - Healthy P1+B1")
w1+w2+w3

In [ ]:
params$sample_id

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, 
                                 "_WNN_RNA_ATAC-clustering-comparison.png")), w1 + w2 + w3, width = 30, height = 10, units = "in")

In [ ]:
w

In [ ]:
# plot RNA, ATAC and WNN based on orig.ident
wo1 <- DimPlot(w, reduction = "umap.harmony.SCT", label = FALSE, label.size = 3, repel = TRUE, group.by = "orig.ident2", shuffle = TRUE) + ggtitle("Integrated RNA - Healthy P1+B1")
wo2 <- DimPlot(w, reduction = "umap.reducedpeaks.integrated",  label = FALSE, label.size = 3, repel = TRUE, group.by = "orig.ident2", shuffle = TRUE) + ggtitle("Integrated ATAC - Healthy P1+B1")
wo3 <- DimPlot(w, reduction = "jointumap", label = FALSE, label.size = 3, repel = TRUE, group.by = "orig.ident2", shuffle = TRUE) + ggtitle("Integrated WNN - Healthy P1+B1")
wo1+wo2+wo3

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_WNN_RNA_ATAC-origident-comparison.png")), wo1+wo2+wo3, width = 30, height = 10, units = "in")

In [ ]:
# save all together
ggsave(file.path(fig_dir, paste0(params$sample_id, "_WNN_RNA_ATAC-umap-comparisons.png")), w1+w2+w3 + wo1+wo2+wo3, width = 30, height = 10, units = "in")

In [ ]:
# Plot diff resolutions for WNN
w@meta.data$wsnn_res.0.2 <- factor(w@meta.data$wsnn_res.0.2, levels = as.character(0:max(as.numeric(w@meta.data$wsnn_res.0.2))))
w@meta.data$wsnn_res.0.5 <- factor(w@meta.data$wsnn_res.0.5, levels = as.character(0:max(as.numeric(w@meta.data$wsnn_res.0.5))))
w@meta.data$wsnn_res.0.8 <- factor(w@meta.data$wsnn_res.0.8, levels = as.character(0:max(as.numeric(w@meta.data$wsnn_res.0.8))))

wc1 <- DimPlot(w, reduction = 'jointumap', group.by = "wsnn_res.0.2", label = TRUE, pt.size = 0.5) + labs(title = "WNN - res.0.2 - Healthy P1+B1")
wc2 <- DimPlot(w, reduction = 'jointumap', group.by = "wsnn_res.0.5", label = TRUE, pt.size = 0.5) + labs(title = "WNN - res.0.5 - Healthy P1+B1")
wc3 <- DimPlot(w, reduction = 'jointumap', group.by = "wsnn_res.0.8", label = TRUE, pt.size = 0.5) + labs(title = "WNN- res.0.8 - Healthy P1+B1")
wc4 <- DimPlot(w, reduction = 'jointumap', group.by = "wsnn_res.1", label = TRUE, pt.size = 0.5) + labs(title = "WNN - res.1 - Healthy P1+B1")
wc1 + wc2 + wc3 + wc4

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_WNN_Cluster-Resolutions.png")), wc1 + wc2 + wc3 + wc4, width = 20, height = 15, units = "in")

# **Cell Annotation**

In [ ]:
w <- readRDS(here(so_dir, paste0(params$sample_sequencing_id, "_seurat-wnn-integrated2.RDS")))

In [ ]:
DefaultAssay(w) <- "SCT"

In [ ]:
Idents(w) <- "wsnn_res.1"
unique(Idents(w))

In [ ]:
features <- c("Atp4a", "Atp4b", "Mfsd4a", #parietal
              "Muc5ac", "Capn8", # foveolar
              "Muc6", #neck
              "Cblif", "Pgc", #chief
              "Dclk1", # tuft 
              "Krt4", "Krt5", # squamous
              "Sst", # dcells
              "Gast", # gcells
              "Ghrl" ,"Chga", "Hdc", #enteroendocrine
              "Mki67", "Stmn1", "Lgr5", "Tff2", #dividing,
              "Pdgfra", "Cald1", "Lama2", "Robo2", "Cd34", # fb
              "Pecam1", "Cdh5" #endothelial
              )

In [ ]:
dp <- DotPlot(w, 
        assay = "SCT",
        features = features) + RotatedAxis()

dp

ggsave(file.path(fig_dir, paste0(params$sample_id, "_GeneExpression_MainMarkers_final.png")), dp, width = 15, height = 15, units = "in")

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_GeneExpression_MainMarkers_final.png")), dp, width = 15, height = 15, units = "in")

In [ ]:
# save
saveRDS(w, here(so_dir, paste0(params$sample_id, "_seurat-wnn-integrated_annotated.RDS")))

### Pseudobulking to find Markers across clusters

In [ ]:
w_aggregated <- AggregateExpression(  
  w, 
  assay = "SCT", 
  slot = "counts",                                               # data slot has log normalized data 
  group.by = c("HTO_origin", "wsnn_res.1"), 
  return.seurat = TRUE)

In [ ]:
# extract information from Cells to add to the metadata

w_aggregated$sample.origin.detail <- sapply(strsplit(Cells(w_aggregated), "_"), function(x) x[1])

In [ ]:
library(DESeq2)

In [ ]:
# Perform DE analysis across clusters
unique(Idents(w_aggregated))

Idents(w_aggregated) <- "seurat_clusters"
Idents(w_aggregated)

In [ ]:
pseudobulk_WNN_markers_CS <- FindAllMarkers(
  object = w_aggregated, 
  assay = "SCT", 
  slot = "data", # this slot contains corrected counts (no effect of library size) # https://github.com/satijalab/seurat/discussions/4210 # https://github.com/satijalab/seurat/issues/5751
  only.pos = FALSE, # Only return positive markers
  test.use = "DESeq2", # Using DESeq2 for differential expression
  min.pct = 0.1, # Minimum fraction of cells expressing the gene
  logfc.threshold = 0.25 # Log fold change threshold
)


In [ ]:
write.csv(pseudobulk_WNN_markers_CS, row.names = FALSE,
          here(table_dir, paste0(params$sample_id, "_pseudobulk_WNN_markers_clusters.csv")))


saveRDS(w_aggregated, here(so_dir, paste0(params$sample_id, "_seurat-wnn-pseudobulk-aggregated.RDS")))


### Plot the top 5 DEG for specific clusters 

In [ ]:
library(dplyr)

top_genes_table <- marker_data_cluster %>%
  filter(cluster %in% clusters_of_interest) %>%
  group_by(cluster) %>%
  top_n(n = 5, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC)) %>%
  select(cluster, gene, avg_log2FC)

top_genes_table

In [ ]:
Idents(w) <- "wsnn_res.1"
w <- RenameIdents(w,
                          "0" = "Chief",
                          "1" = "Foveolar", 
                          "2" = "Neck", 
                          "3" = "Parietal", 
                          "4" = "Chief", 
                          "5" = "Isthmus_5", 
                          "6" = "Chief", 
                          "7" = "Parietal", 
                          "8" = "Foveolar", 
                          "9" = "Foveolar", 
                          "10" = "Parietal", 
                          "11" = "Neck", 
                          "12" = "Isthmus_12", 
                          "13" = "Foveolar", 
                          "14" = "Fb_14", 
                          "15" = "Chga_Hdc", 
                          "16" = "Dividing", 
                          "17" = "Fb_17", 
                          "18" = "C_18", 
                          "19" = "Parietal", 
                          "20" = "Isthmus_20", 
                          "21" = "Ghrl", 
                          "22" = "D_cells", 
                          "23" = "Fb_23", 
                          "24" = "Tuft", 
                          "25" = "Parietal_25",
                          "26" = "Endothelial", 
                          "27" = "Fb_27", 
                          "28" = "Neural", 
                          "29" = "Lymphatic", 
                          "30" = "Immune",
                          "31" = "Squamous_Epi",
                          "32" = "Fb_32",
                          "33" = "Endothelial",
                          "34" = "Fb_34",
                          "35" = "Liver_metabolism",
                          "36" = "Neural_2",
                          "37" = "Adipocytes") 


In [ ]:
w <- AddMetaData(object = w, 
                 metadata = Idents(w), 
                 col.name = 'Annotated')

In [ ]:
before2 <- DimPlot(w, reduction = 'jointumap', group.by = "wsnn_res.1", label = TRUE, pt.size = 0.5) + labs(title = "WNN - res.1 - Healthy P1+B1")
after2  <- DimPlot(w, reduction = 'jointumap', group.by = "Annotated", label = TRUE, pt.size = 0.5) + labs(title = "WNN - res.1 - Healthy P1+B1")

before2 + after2

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_WNN_Cluster-Annotation.png")), before + after, width = 20, height = 10, units = "in")

In [ ]:
# save
saveRDS(w, here(so_dir, paste0(params$sample_id, "_seurat-wnn-integrated_annotated.RDS")))

In [ ]:
chief_markers <- c("Cblif", "Pgc")

In [ ]:
w <- AddModuleScore(w, features = list(chief_markers), name = "chief_markers")

In [ ]:
w

In [ ]:
head(w@meta.data)

In [ ]:
chief_exp <- FeaturePlot(w, c("chief_markers1"), order = FALSE, reduction = "jointumap") + ggtitle("Chief Markers \n Cblif, Pgc")
chief_exp

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_Chief.png")), chief_exp, width = 10, height = 10, units = "in")

In [ ]:
foveolar_markers <- c("Muc5ac", "Capn8")
w <- AddModuleScore(w, features = list(foveolar_markers), name = "foveolar_markers")

In [ ]:
foveolar_exp <- FeaturePlot(w, c("foveolar_markers1"), order = FALSE, reduction = "jointumap") + ggtitle("Foveolar Markers \n Muc5ac, Capn8")
foveolar_exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_Foveolar.png")), foveolar_exp, width = 10, height = 10, units = "in")

In [ ]:
parietal_markers <- c("Atp4a", "Atp4b", "Mfsd4a")
w <- AddModuleScore(w, features = list(parietal_markers), name = "parietal_markers")

In [ ]:
parietal_exp <- FeaturePlot(w, c("parietal_markers1"), order = FALSE, reduction = "jointumap") + ggtitle("Parietal Markers \n Atp4a, Atp4b, Mfsd4a")
parietal_exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_Parietal.png")), parietal_exp, width = 10, height = 10, units = "in")

In [ ]:
neck_markers <- c("Muc6")
w <- AddModuleScore(w, features = list(neck_markers), name = "neck_markers")

In [ ]:
neck_exp <- FeaturePlot(w, c("neck_markers1"), order = FALSE, reduction = "jointumap") + ggtitle("Neck Markers \n Muc6")
neck_exp

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_Neck.png")), neck_exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "endothelial_markers"
markers <- c("Pecam1", "Cdh5")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp

In [ ]:
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "tuft_markers"
markers <- c("Dclk1")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "enteroendocrine_markers"
markers <- c("Ghrl", "Chga", "Hdc")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "dividing_markers"
markers <- c("Mki67", "Stmn1")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "dividing_marker"
markers <- c("Mki67")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "G_markers"
markers <- c("Gast")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
# how many cells expressing gast?

gene_name <- "Gast"  
expr_values <- FetchData(w, vars = gene_name)
num_expressing <- sum(expr_values[,1] > 0)

print(num_expressing)

In [ ]:
gene_name <- "Gast"  # replace with your gene of interest
expr_values <- FetchData(w, vars = gene_name, assay = "RNA", layer = "counts")
num_expressing <- sum(expr_values[,1] > 0)
print(num_expressing)

In [ ]:
cell_type <- "D_markers"
markers <- c("Sst")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "immune_markers"
markers <- c("Ptprc", "Ms4a6b")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "squamous_markers"
markers <- c("Krt13", "Krt14")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "lymphatic_markers"
markers <- c("Mmrn1", "Reln")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")

In [ ]:
cell_type <- "fibroblast_markers"
markers <- c("Pdgfra", "Cald1", "Lama2", "Robo2", "Cd34", "Acta2", "Actg2", "Myl9")
w <- AddModuleScore(w, features = list(markers), name = cell_type)
exp <- FeaturePlot(w, 
                   features = paste0(cell_type,"1"), 
                   order = FALSE, 
                   reduction = "jointumap") + ggtitle(paste0(cell_type, "\n", paste(markers, collapse = ", ")))
exp
ggsave(file.path(fig_dir, paste0(params$sample_id, "_MarkerExp_", cell_type, ".png")), exp, width = 10, height = 10, units = "in")